# Day 4 - Qwen3-VL

- status: implemented_toy_not_executed_real_model
- stage: VLM_DAY_4
- paper_ids: qwen3_vl_2025
- dataset_ids: synthetic_toy, user_selected_nas_data
- seed: 42
- scope: educational implementation; real inference/training is opt-in

이 노트북은 다른 사용자 파일, 공유 환경, checkpoint를 자동으로 변경하지 않는다. 실제 데이터는
`/nas/datahub/min` 아래 사용자가 지정한 경로만 읽는다.

## 1. Learning question

Qwen3-VL에서 image가 어떤 경로로 language generation에 들어가며 Instruct/Thinking, dense/MoE, 2B/8B 선택은 무엇을 바꾸는가?

## 2. Background theory

`dynamic-resolution image -> ViT -> merger/projector -> visual embeddings interleaved with text -> Qwen LLM -> autoregressive tokens`. Qwen3-VL의 주요 갱신은 Interleaved-MRoPE, multi-level ViT feature를 쓰는 DeepStack, video text-timestamp alignment다.

## 3. Paper connection

Qwen3-VL은 dense와 MoE, Instruct와 Thinking 변형을 제공한다. 이 과정의 기본은 작은 실습에 적합한 `Qwen/Qwen3-VL-2B-Instruct`이며 전체 benchmark 재현을 목표로 하지 않는다.

## 4. Input/output and shapes

입력은 role/content 형태의 multimodal chat message다. processor가 image token과 text token을 만들고 model.generate가 prompt 뒤 output token을 생성한다. 응답은 free-form text이므로 task별 schema 검증이 필요하다.

In [ ]:
from pathlib import Path
import sys
import numpy as np

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "pyproject.toml").is_file()),
    Path("/nas/home/mhlee/vlm-foundation-7days"),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("project:", PROJECT_ROOT)
print("numpy:", np.__version__)

## 5. Minimal implementation

공식 `apply_chat_template`와 동일한 message contract를 구성하되 모델을 로드하지 않는다.

In [ ]:
from pathlib import Path

image_path = Path("/nas/datahub/min/<user-selected>/images/example.jpg")
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image_path.as_uri()},
        {"type": "text", "text": "Describe only visible facts in one sentence."},
    ],
}]
print(messages)

## 6. Visualization sanity check

message에서 image와 text의 순서, file URI, generation prompt 존재를 확인한다.

In [ ]:
assert messages[0]["content"][0]["type"] == "image"
assert messages[0]["content"][1]["type"] == "text"
print("message contract: PASS")

## 7. Experiment

Instruct prompt를 짧은 caption, OCR JSON, grounding JSON으로 바꿔 output contract 차이를 설계한다.

In [ ]:
task_prompts = {
    "caption": "Describe only visible facts in one sentence.",
    "ocr": 'Return only JSON: [{"text":"...","bbox_2d":[x1,y1,x2,y2]}]',
    "grounding": 'Return only JSON: [{"label":"helmet","bbox_2d":[x1,y1,x2,y2]}]',
}
print(task_prompts)

## 8. Metrics

task별 accuracy 외에 input/output token, TTFT, tokens/sec, peak memory, JSON validity, hallucination rate를 기록한다.

## 9. Interpretation

VLM은 detector처럼 고정 tensor를 반환하지 않고 language token을 생성한다. 유연성은 높지만 schema failure와 hallucination이 새 오류 축이 된다.

## 10. Failure cases

chat template 누락, prompt token trim 오류, unsupported transformers version, cache miss, multi-image 순서 혼동, 너무 긴 visual context를 확인한다.

## 11. Real-service implications

model process는 시작 시 한 번 load하고 요청마다 message/visual token budget을 검증한다. free-form 응답은 JSON parser와 retry/fallback 경계를 거친다.

## 12. Review questions

1. Interleaved-MRoPE가 표현하는 축은?
2. DeepStack은 어느 정보를 LLM에 더 제공하는가?
3. Instruct와 Thinking의 latency/출력 차이는?
4. free-form generation이 detector보다 만드는 새 위험은?
5. 2B와 8B를 비교할 때 고정해야 할 조건은?